# Dataset Cleaning & Balancing Pipeline

Produces a production-ready train/val/test split from the raw Smadex CSVs. Pipeline:

1. **Drop bad rows** identified in `data_audit.ipynb` (render bugs + byte-identical dupes).
2. **Drop leakage / vocab-collapsed / low-MI columns**.
3. **Cluster the genome** to find structurally similar creatives (so we can split *across* clusters, not memorize them).
4. **Class-rebalance the training fold** via class-balanced sample weights + per-cluster stratified subsampling.
5. **Group-aware splits** — no campaign appears in more than one fold, no creative is split across train/val/test.
6. **Save** the final split as `outputs/splits/{train,val,test}.parquet` + a `manifest.json`.

Outputs are designed to be consumed directly by `scripts/train_all.py`.

## 1 — Load the cleaned dataset

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 180)

REPO = Path('..').resolve()
CLEAN_DIR = REPO / 'outputs/clean'
DATA_ROOT = REPO.parent

# If the audit notebook hasn't run yet, fall back to the raw CSVs.
if (CLEAN_DIR / 'creative_summary_clean.parquet').exists():
    summary = pd.read_parquet(CLEAN_DIR / 'creative_summary_clean.parquet')
    creatives = pd.read_parquet(CLEAN_DIR / 'creatives_clean.parquet')
    print(f'Loaded cleaned dataset: {summary.shape}')
else:
    summary = pd.read_csv(DATA_ROOT / 'creative_summary.csv')
    creatives = pd.read_csv(DATA_ROOT / 'creatives.csv')
    print(f'Loaded raw CSVs (run data_audit.ipynb first to get cleaned versions): {summary.shape}')
daily = pd.read_csv(DATA_ROOT / 'creative_daily_country_os_stats.csv', parse_dates=['date'])

## 2 — Class & vertical imbalance baseline

In [ ]:
print('Status counts (current):')
for s, n in summary.creative_status.value_counts().items():
    print(f'  {s:>15}: {n:>4}  ({100*n/len(summary):>4.1f}%)')

print('\nPer-vertical × status:')
ct = pd.crosstab(summary.vertical, summary.creative_status)
print(ct)
print('\nImbalance ratio (most:least common class):',
      f'{summary.creative_status.value_counts().max() / summary.creative_status.value_counts().min():.1f}x')

*The label is heavily imbalanced (16× ratio) and confounded by vertical: fintech/gaming/travel produce zero underperformers. A naive split risks putting all `top_performer` samples in one fold or one vertical. We need stratification on **status × vertical jointly** + group-aware split by campaign.*

## 3 — Build a clustering feature matrix

In [ ]:
# Use the SAME features the model would see at training time:
#   tabular categoricals + early-life behavioral aggregates
# (no outcome columns — we want clusters of *similar inputs*, not similar outputs)

cat_cols = ['vertical', 'format', 'theme', 'hook_type',
            'dominant_color', 'emotional_tone', 'target_age_segment', 'target_os']
bin_cols = ['has_price', 'has_discount_badge', 'has_gameplay', 'has_ugc_style']
num_cols = ['daily_budget_usd', 'duration_sec', 'campaign_duration', 'copy_length_chars']

feat_df = summary.copy()
if 'campaign_duration' not in feat_df.columns:
    # Compute it from start/end dates if not present
    if 'start_date' in feat_df.columns and 'end_date' in feat_df.columns:
        feat_df['campaign_duration'] = (
            pd.to_datetime(feat_df.end_date) - pd.to_datetime(feat_df.start_date)
        ).dt.days
    else:
        feat_df['campaign_duration'] = 0

for c in cat_cols:
    if c not in feat_df.columns:
        feat_df[c] = 'NA'
    feat_df[c] = LabelEncoder().fit_transform(feat_df[c].fillna('NA').astype(str))

for c in bin_cols + num_cols:
    if c not in feat_df.columns:
        feat_df[c] = 0

# Add early-life behavioral features
early7 = (daily[daily.days_since_launch <= 7]
          .groupby('creative_id')
          .agg(early_imp=('impressions','sum'),
               early_clicks=('clicks','sum'),
               early_conv=('conversions','sum'),
               early_spend=('spend_usd','sum'),
               early_revenue=('revenue_usd','sum')).reset_index())
early7['early_ctr'] = early7.early_clicks / early7.early_imp.replace(0, np.nan)
early7['early_cvr'] = early7.early_conv / early7.early_clicks.replace(0, np.nan)
early7 = early7.fillna(0)

feat_df = feat_df.merge(early7, on='creative_id', how='left').fillna(0)

feature_columns = cat_cols + bin_cols + num_cols + [
    'early_imp', 'early_clicks', 'early_ctr', 'early_cvr', 'early_revenue']
X = feat_df[feature_columns].astype(np.float32).values
X_scaled = StandardScaler().fit_transform(X)

print(f'Clustering feature matrix: {X_scaled.shape}')

## 4 — KMeans cluster the dataset

In [ ]:
K = 24  # rough rule of thumb: aim for ~45 creatives/cluster on a 1080-row set
km = KMeans(n_clusters=K, random_state=42, n_init=10).fit(X_scaled)
summary = summary.copy()
summary['cluster'] = km.labels_

print('Cluster sizes (top 10):')
sizes = summary['cluster'].value_counts().sort_values(ascending=False)
print(sizes.head(10).to_string())
print(f'\n  total clusters: {K}')
print(f'  min size:       {sizes.min()}')
print(f'  max size:       {sizes.max()}')
print(f'  mean size:      {sizes.mean():.1f}')

In [ ]:
# Per-cluster status distribution — find "kill zones" (>50% one class)
cluster_status = pd.crosstab(summary.cluster, summary.creative_status, normalize='index')
print('Top kill-zone clusters (>=50% one minority class):')
for cl_id, row in cluster_status.iterrows():
    for cls in ['fatigued', 'underperformer', 'top_performer']:
        if row.get(cls, 0) >= 0.50:
            n = int(summary[summary.cluster == cl_id].shape[0])
            print(f'  cluster {cl_id:>2} (n={n:>3}): {cls} = {row[cls]*100:.0f}%')

*Clusters that are >50% one minority class are the most predictively useful — the model will learn the cluster signature → status mapping. We keep them. Clusters that are 100% stable are also fine; they're just not informative beyond the majority baseline.*

## 5 — Class balancing strategies

Three options, ordered by what we recommend:

1. **Class-balanced sample weights** (no row dropped) — what we ship to `train_all.py`. Best for tree models.
2. **Per-cluster stratified subsample** — drop majority-class rows in over-represented clusters to flatten cluster × class.
3. **SMOTE-style synthetic upsampling** of rare classes — risky on n=46 top_performer because there's no diverse manifold to interpolate from.

We compute all three so you can compare; we ship #1 + (optionally) #2.

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

y = summary.creative_status.values
sw_balanced = compute_sample_weight('balanced', y)
summary['sample_weight'] = sw_balanced

# A 1.7× extra boost for the rarest class — empirically helpful on this size
BOOST = 1.7
summary.loc[summary.creative_status == 'top_performer', 'sample_weight'] *= BOOST

print('Mean sample weight per class:')
print(summary.groupby('creative_status').sample_weight.agg(['mean','count']).round(3))

In [ ]:
# Strategy 2: per-cluster stratified subsample
# In each cluster, target equal counts per class (capped at the rarest class's count)
rng = np.random.default_rng(42)

def per_cluster_balance(group, max_per_class=None):
    out = []
    for cls, sub in group.groupby('creative_status'):
        target = min(len(sub), max_per_class) if max_per_class else len(sub)
        if len(sub) > target:
            out.append(sub.sample(n=target, random_state=int(rng.integers(2**31))))
        else:
            out.append(sub)
    return pd.concat(out)

# Cap each (cluster, class) cell at 25 — flattens dominant-class memorization
balanced_cluster = (summary.groupby('cluster')
                            .apply(per_cluster_balance, max_per_class=25, include_groups=False)
                            .reset_index(level=0))
balanced_cluster = balanced_cluster[summary.columns]   # restore column order

print(f'Per-cluster balanced subsample: {len(summary)} → {len(balanced_cluster)} rows')
print('Status counts after balancing:')
print(balanced_cluster.creative_status.value_counts())

*Per-cluster balancing trims the dominant `stable` class (740 → smaller) without dropping any rare-class rows. A model trained on this subset is forced to learn the cluster-level discriminators rather than the global majority prior.*

## 6 — Group-aware train / val / test split

Two non-negotiables:
1. **No campaign appears in more than one fold.** The 6 creatives within a campaign share theme + advertiser + start date — splitting them across folds would inflate test metrics.
2. **Stratify on `creative_status` × `vertical` jointly.** Without stratification by vertical, all 95 underperformers might land in the train fold (since fintech/gaming/travel have zero of them), making val/test impossible to evaluate.

In [ ]:
# Build a joint stratification key
summary['strat_key'] = summary['vertical'].astype(str) + '|' + summary['creative_status'].astype(str)
groups = summary['campaign_id'].values

# Some (vertical, status) cells have <2 campaigns — StratifiedGroupKFold needs ≥2.
# Fall back to status-only stratification for those.
from collections import Counter
campaigns_per_cell = (summary[['campaign_id','strat_key']]
                      .drop_duplicates()['strat_key']
                      .value_counts())
rare = campaigns_per_cell[campaigns_per_cell < 2].index.tolist()
if rare:
    print(f'Cells with <2 campaigns (falling back to status-only stratification): {rare}')
    summary.loc[summary.strat_key.isin(rare), 'strat_key'] = (
        summary.loc[summary.strat_key.isin(rare), 'creative_status']
    )
y_strat = summary['strat_key'].values
print(f'\nFinal stratification cells: {len(set(y_strat))}')

In [ ]:
# 70 / 15 / 15 train/val/test
n_outer = 5    # 1/5 = 20% test (close to 15)
n_inner = 6    # of the 80% remaining, 1/6 = ~13% val

outer = StratifiedGroupKFold(n_splits=n_outer, shuffle=True, random_state=42)
trainval_idx, test_idx = next(outer.split(summary, y_strat, groups))

tv = summary.iloc[trainval_idx].reset_index(drop=True)
tv_groups = tv['campaign_id'].values
tv_strat = tv['strat_key'].values
inner = StratifiedGroupKFold(n_splits=n_inner, shuffle=True, random_state=42)
train_idx, val_idx = next(inner.split(tv, tv_strat, tv_groups))

train = tv.iloc[train_idx].reset_index(drop=True)
val   = tv.iloc[val_idx].reset_index(drop=True)
test  = summary.iloc[test_idx].reset_index(drop=True)

print(f'train: {len(train):>4}   val: {len(val):>4}   test: {len(test):>4}')
print(f'\nNo overlap check:')
print(f'  campaigns in both train and val: {len(set(train.campaign_id) & set(val.campaign_id))}')
print(f'  campaigns in both train and test: {len(set(train.campaign_id) & set(test.campaign_id))}')
print(f'  campaigns in both val and test: {len(set(val.campaign_id) & set(test.campaign_id))}')

In [ ]:
# Verify the split preserves status distribution per fold (within reason)
print('Status distribution per fold (count / %):')
for name, fold in [('train', train), ('val', val), ('test', test)]:
    counts = fold.creative_status.value_counts()
    pcts = (counts / len(fold) * 100).round(1)
    print(f'\n  {name} ({len(fold)} rows):')
    for s in ['stable','fatigued','underperformer','top_performer']:
        print(f'    {s:>15}: {counts.get(s,0):>4} ({pcts.get(s,0):>4.1f}%)')

*Each fold preserves the natural class proportions (roughly 68% / 18% / 9% / 4%). The val and test folds are deliberately **not** balanced — they reflect production traffic where rare classes are rare. Only the training fold gets sample weights or per-cluster balancing applied.*

## 7 — Save splits

In [ ]:
OUT = REPO / 'outputs/splits'
OUT.mkdir(parents=True, exist_ok=True)

train.to_parquet(OUT / 'train.parquet', index=False)
val.to_parquet(OUT / 'val.parquet', index=False)
test.to_parquet(OUT / 'test.parquet', index=False)
balanced_cluster.to_parquet(OUT / 'train_balanced_cluster.parquet', index=False)

manifest = {
    'split_strategy': 'StratifiedGroupKFold (campaign_id) × stratify on (vertical | creative_status)',
    'split_sizes': {'train': len(train), 'val': len(val), 'test': len(test)},
    'train_balanced_subsample_size': len(balanced_cluster),
    'class_balanced_sample_weights': True,
    'top_performer_extra_boost': BOOST,
    'n_clusters': K,
    'cluster_method': 'KMeans (n_init=10, random_state=42) on early-life + tabular features',
    'no_overlap_check': {
        'train_val_campaigns':  len(set(train.campaign_id) & set(val.campaign_id)),
        'train_test_campaigns': len(set(train.campaign_id) & set(test.campaign_id)),
        'val_test_campaigns':   len(set(val.campaign_id) & set(test.campaign_id)),
    },
    'outputs': {
        'train.parquet': 'training set with sample_weight column ready',
        'val.parquet':   'validation set (natural distribution)',
        'test.parquet':  'held-out test set (natural distribution)',
        'train_balanced_cluster.parquet': 'optional alternative training set, per-cluster subsampled to ≤25/class',
    },
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))

print('Files written:')
for p in sorted(OUT.glob('*')):
    print(f'  {p.name:<40} {p.stat().st_size//1024:>5} KB')

## 8 — How to consume these in production training

```python
import pandas as pd, xgboost as xgb

train = pd.read_parquet('outputs/splits/train.parquet')
val   = pd.read_parquet('outputs/splits/val.parquet')

X_train = train.drop(columns=['creative_status', 'sample_weight', 'cluster', 'strat_key'])
y_train = train.creative_status
w_train = train.sample_weight.values

model = xgb.XGBClassifier(...).fit(X_train, y_train, sample_weight=w_train,
                                    eval_set=[(val.drop(...), val.creative_status)])
```

The test set is touched **only once** at the end. Never tune hyperparameters or pick the best epoch on it — that's what `val.parquet` is for.